# EasyVitessce Example: SpatialData-Plot with MERFISH dataset

## Downloading and importing necessary packages

By default, interactive plots are enabled upon importing easy_vitessce. This notebook aims to demonstrate the transition between static and interactive plots, so the interactive plots are initially turned off.

In [ ]:
#!pip install easy_vitessce
#!pip install spatialdata
#!pip install spatialdata_plot

In [ ]:
import easy_vitessce as ev 
import spatialdata as sd
import spatialdata_plot
from os.path import join

In [ ]:
from vitessce.data_utils import (
    sdata_morton_sort_points,
    sdata_points_process_columns,
    sdata_points_modify_row_group_size,
    sdata_morton_query_rect,
)

In [ ]:
# Enable interactive plots
ev.configure_plots(enable_plots=["spatialdata-plot"])

## Download the data

In [ ]:
import os
from os.path import join, isfile, isdir
from urllib.request import urlretrieve
import zipfile

In [ ]:
data_dir = "data"
zip_path = join(data_dir, "xenium_rep1_io.spatialdata.zarr.zip")
sdata_path = join(data_dir, "xenium_rep1_io.spatialdata.zarr")

In [ ]:
if not isdir(sdata_path):
    if not isfile(zip_path):
        os.makedirs(data_dir, exist_ok=True)
        urlretrieve('https://data-2.vitessce.io/sdata-datasets/xenium_rep1_io.spatialdata.zarr.zip', zip_path)
    with zipfile.ZipFile(zip_path,"r") as zip_ref:
        zip_ref.extractall(data_dir)
        os.rename(join(data_dir, "data.zarr"), sdata_path)

## Read the data

In [ ]:
sdata = sd.read_zarr(sdata_path)
sdata

## Make the points ready for tiled access

References:
- https://vitessce.io/docs/data-troubleshooting/#points
- https://github.com/vitessce/vitessce-python/blob/main/docs/notebooks/spatial_data_xenium_morton.ipynb

In [ ]:
if "transcripts_with_morton_codes" not in sdata.points:
    transformations = sdata['transcripts'].attrs['transform']
    
    sdata = sdata_morton_sort_points(sdata, "transcripts")
    
    # Add feature_index column to dataframe, and reorder columns so that feature_name (dict column) is the rightmost column.
    ddf = sdata_points_process_columns(sdata, "transcripts", var_name_col="feature_name", table_name="table")
    
    del ddf.attrs['transform']
    sdata["transcripts_with_morton_codes"] = PointsModel.parse(ddf, feature_key="feature_name", instance_key="cell_id", transformations=transformations)
    sdata.write_element("transcripts_with_morton_codes")
    
    sdata_points_modify_row_group_size(sdata, "transcripts_with_morton_codes", row_group_size=25_000)

## Static plotting

In [ ]:
sdata

In [ ]:
sdata.points["transcripts_with_morton_codes"]

In [ ]:
sdata.points["transcripts_with_morton_codes"].attrs

In [ ]:
sdata

In [ ]:
vw = (
    sdata
        .pl.render_images(element="morphology_focus")
        .pl.render_shapes(element="cell_boundaries", table_name="table")
        .pl.render_points(element="transcripts_with_morton_codes", color="feature_name", groups=["ERBB2", "ACTA2"], palette=["red", "green"], table_name="table")
        .pl.show()
)
vw

In [ ]:
vw.config.to_dict(base_url="")

In [ ]:
# add another code block for the mouse liver dataset? 

## Deactivating interactive plots

In [ ]:
# Disable interactive plots
ev.configure_plots(disable_plots=["spatialdata-plot"])

## Static plotting

In [ ]:
vw = (
    sdata
        .pl.render_images(element="morphology_focus")
        .pl.render_shapes(element="cell_boundaries", table_name="table")
        .pl.render_points(element="transcripts_with_morton_codes", color="feature_name", groups=["ERBB2", "ACTA2"], palette=["red", "green"], table_name="table")
        .pl.show()
)
vw